In [0]:
!pip install kafka-python

In [0]:
import json
from logging import config
import os
import random
import uuid
import time
from datetime import datetime, timezone
from kafka import KafkaProducer

def load_properties(filepath):
    props = {}
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            key, value = line.split("=", 1)
            props[key.strip()] = value.strip()

    return props

config = load_properties("client.properties")

KAFKA_BROKER = config["bootstrap.servers"]
USER_NAME = config["sasl.username"]
PWD = config["sasl.password"]
TOPIC = config["sasl.topic"]


producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    acks=1,
    linger_ms=5,
    batch_size=32768,
    security_protocol="SASL_SSL",
    sasl_mechanism="PLAIN",
    sasl_plain_username=USER_NAME,
    sasl_plain_password=PWD
)

HYD_LAT = 17.3850
HYD_LON = 78.4867

ride_statuses = [
    "requested",
    "accepted",
    "started",
    "completed",
    "cancelled"
]

def random_coord(base, spread=0.15):
    return round(base + random.uniform(-spread, spread), 6)

def generate_ride():
    return {
        "ride_id": str(uuid.uuid4()),
        "driver_id": f"DRV{random.randint(1000, 9999)}",
        "rider_id": f"RID{random.randint(10000, 99999)}",
        "pickup_lat": random_coord(HYD_LAT),
        "pickup_lon": random_coord(HYD_LON),
        "drop_lat": random_coord(HYD_LAT),
        "drop_lon": random_coord(HYD_LON),
        "fare": round(random.uniform(50, 1500), 2),
        "distance": round(random.uniform(1, 50), 2),
        "ride_status": random.choice(ride_statuses),
        "event_time": datetime.now(timezone.utc).isoformat()
    }

TARGET_RATE = 1000
BATCH_SIZE = 100

while True:
    start = time.time()

    for _ in range(TARGET_RATE):
        producer.send(TOPIC, generate_ride())
    producer.flush()

    elapsed = time.time() - start

    if elapsed < 5:
        time.sleep(5 - elapsed)

    print(f"Produced ~{TARGET_RATE} events/sec")